In [1]:
# ── Walk‑Forward Kelly Simulation (de‑vigged features for inference,
#    VIGGED moneylines used for payout) ──────────────────────────────
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════════
# 0. Power‑method de‑vigging (same as app & original simulation)
# ═══════════════════════════════════════════════════════════════════════════
def power_de_vig(home_implied, away_implied, max_iter=100, tol=1e-12):
    """
    Remove the overround using the power (multiplicative) method.
    Finds a scalar k such that:
        home_implied^k + away_implied^k = 1
    and returns fair_home, fair_away.
    """
    home_implied = np.asarray(home_implied, dtype=float)
    away_implied = np.asarray(away_implied, dtype=float)
    n = len(home_implied)
    fair_home = np.full(n, np.nan)
    fair_away = np.full(n, np.nan)

    for i in range(n):
        h = home_implied[i]
        a = away_implied[i]
        if np.isnan(h) or np.isnan(a) or h <= 0 or a <= 0:
            continue
        if abs(h + a - 1.0) < tol:
            fair_home[i], fair_away[i] = h, a
            continue

        def f(k):
            return h**k + a**k - 1.0
        def fp(k):
            if h > 0 and a > 0:
                return h**k * np.log(h) + a**k * np.log(a)
            return 0.0

        k = 1.0
        converged = False
        for _ in range(max_iter):
            fk = f(k)
            if abs(fk) < tol:
                converged = True
                break
            fpk = fp(k)
            if fpk == 0:
                break
            k_new = k - fk / fpk
            if k_new <= 0: k_new = 0.001
            if k_new > 10.0: k_new = 10.0
            if abs(k_new - k) < tol:
                k = k_new; converged = True; break
            k = k_new

        if not converged:
            lo, hi = 0.001, 1.0
            if f(lo) < 0: lo, hi = hi, lo
            for _ in range(max_iter):
                mid = (lo + hi) / 2
                if f(mid) == 0.0 or (hi - lo) / 2 < tol:
                    k = mid; converged = True; break
                if np.sign(f(mid)) == np.sign(f(lo)):
                    lo = mid
                else:
                    hi = mid
            if not converged:
                total = h + a
                fair_home[i] = h / total if total > 0 else np.nan
                fair_away[i] = a / total if total > 0 else np.nan
                continue

        fair_home[i] = h**k
        fair_away[i] = a**k
    return fair_home, fair_away

# ═══════════════════════════════════════════════════════════════════════════
# 1. Load model bundle & data
# ═══════════════════════════════════════════════════════════════════════════
bundle = joblib.load("nba.pkl")
model = bundle["model"]
scaler = bundle["scaler"]
selected_features = bundle["features"]

INITIAL_BANKROLL = 10000.0
MAX_DRAWDOWN_LIMIT = -0.70          # -70%

df = pd.read_csv("../data/csv/dataset.csv")

# ═══════════════════════════════════════════════════════════════════════════
# 2. Feature engineering (raw → de‑vigged, EXACTLY like the app,
#    BUT ALSO stores the VIGGED decimal odds for reward calculation)
# ═══════════════════════════════════════════════════════════════════════════
def feature_engineering_app_style(df):
    """Replicates the app's inference pipeline:
       1. Parse odds to raw implied prob.
       2. De‑vig using power method (overwrites raw implied probs).
       3. Build diff/ratio features from the FAIR probs.
       Also retains the original vigged moneyline as decimal odds
       (home_decimal_odds, away_decimal_odds) for realistic payouts.
    """
    df = df.copy()

    # ---- Parse to raw implied prob AND vigged decimal odds ----
    for col in ("home_moneyline", "away_moneyline"):
        if col not in df.columns:
            continue
        clean = (df[col].astype(str)
                 .str.replace("+", "", regex=False)
                 .str.replace(",", "", regex=False)
                 .str.replace(" ", "", regex=False))
        num = pd.to_numeric(clean, errors="coerce")

        # Implied probability (raw, before de‑vigging)
        implied = np.where(
            num >= 100, 100 / (num + 100),
            np.where(
                num <= -100,
                np.abs(num) / (np.abs(num) + 100),
                np.where(
                    (num > 1.0) & (num < 100.0),
                    1.0 / num,
                    np.nan
                )
            )
        )
        df[f"{col}_implied_prob"] = implied

        # Vigged decimal odds for actual payout (NOT to be de‑vigged)
        decimal = np.where(
            num >= 100, num/100.0 + 1.0,
            np.where(
                num <= -100, 100.0 / np.abs(num) + 1.0,
                np.where(
                    (num > 1.0) & (num < 100.0), num, np.nan
                )
            )
        )
        # Store in a column that will NOT be overwritten
        col_name = "home_decimal_odds" if col.startswith("home") else "away_decimal_odds"
        df[col_name] = decimal

    # ---- De‑vig BEFORE feature construction (exactly like the app) ----
    col_h = "home_moneyline_implied_prob"
    col_a = "away_moneyline_implied_prob"
    if col_h in df.columns and col_a in df.columns:
        h_vals = df[col_h].values
        a_vals = df[col_a].values
        fair_h, fair_a = power_de_vig(h_vals, a_vals)
        df[col_h] = fair_h
        df[col_a] = fair_a

    # ---- Derived features from FAIR probs ----
    req = ("home_moneyline_implied_prob", "away_moneyline_implied_prob")
    if all(c in df.columns for c in req):
        df["prob_diff"] = df[req[0]] - df[req[1]]
        df["prob_ratio"] = df[req[0]] / (df[req[1]] + 1e-5)

    if "home_elo" in df.columns and "away_elo" in df.columns:
        df["elo_diff"] = df["home_elo"] - df["away_elo"]
        df["elo_ratio"] = df["home_elo"] / (df["away_elo"] + 1e-5)

    return df

df = feature_engineering_app_style(df)

# Drop rows where both moneylines are missing (keep as many as possible)
df = df.dropna(subset=["home_moneyline_implied_prob", "away_moneyline_implied_prob"]).reset_index(drop=True)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

# ═══════════════════════════════════════════════════════════════════════════
# 3. Generate candidate strategies (identical to earlier search)
# ═══════════════════════════════════════════════════════════════════════════
def generate_strategies():
    strategies = []
    min_edge_options = [0.0, 0.01, 0.02, 0.025, 0.03,0.035, 0.04]
    kelly_fracs = [0.25, 0.375, 0.5, 0.75, 1.0]
    cutoffs = [0.005 + 0.01 * i for i in range(10)]

    # Flat
    for min_e in min_edge_options:
        for f in kelly_fracs:
            name = f"Flat_min{min_e:.3f}_kelly{f}"
            strategies.append((name, [(min_e, np.inf, f)]))

    # Two‑bin
    for min_e in min_edge_options:
        for thr in cutoffs:
            if thr <= min_e: continue
            for f_low in kelly_fracs:
                for f_high in kelly_fracs:
                    name = (f"2Bin_min{min_e:.3f}_lo<{thr:.3f}_"
                            f"fL{f_low}_fH{f_high}")
                    strategies.append((name, [
                        (min_e, thr, f_low),
                        (thr, np.inf, f_high)
                    ]))

    # Three‑bin
    breakpoint_pairs = [
        (0.005, 0.025), (0.015, 0.035), (0.025, 0.045), (0.035, 0.055),
        (0.005, 0.035), (0.015, 0.045), (0.025, 0.055), (0.035, 0.065),
    ]
    for min_e in [0.0, 0.01, 0.02]:
        for (bp1, bp2) in breakpoint_pairs:
            if bp1 <= min_e or bp2 <= bp1: continue
            for f1 in [0.25, 0.375, 0.5]:
                for f2 in [0.25, 0.375, 0.5, 0.75]:
                    for f3 in [0.25, 0.375, 0.5, 0.75, 1.0]:
                        name = (f"3Bin_min{min_e:.3f}_"
                                f"[{min_e:.3f}-{bp1:.3f}]f{f1}_"
                                f"[{bp1:.3f}-{bp2:.3f}]f{f2}_"
                                f"[{bp2:.3f}+]f{f3}")
                        strategies.append((name, [
                            (min_e, bp1, f1),
                            (bp1, bp2, f2),
                            (bp2, np.inf, f3)
                        ]))
    return strategies

strategies = generate_strategies()
print(f"Generated {len(strategies)} strategies.\n")

# ═══════════════════════════════════════════════════════════════════════════
# 4. Simulation function (Kelly betting, uses de‑vigged edge calculation
#    BUT settles bets using VIGGED decimal odds for realistic profit)
# ═══════════════════════════════════════════════════════════════════════════
def simulate_kelly(test_df, y_pred_proba, bankroll, edge_strategy):
    """
    test_df must contain:
      - home/away_moneyline_implied_prob (already de‑vigged, fair probs)
      - home_decimal_odds, away_decimal_odds (original vigged odds)
      - winning_team (0=home, 1=away)
    y_pred_proba: model's predicted probability of "away" win (class 1)
    """
    df = test_df.copy()
    df["model_prob_away"] = y_pred_proba
    df["model_prob_home"] = 1 - y_pred_proba

    # Fair prob is already the de‑vigged value stored in the columns
    df["fair_home_prob"] = df["home_moneyline_implied_prob"]
    df["fair_away_prob"] = df["away_moneyline_implied_prob"]

    # Edge uses the de‑vigged probabilities
    df["home_edge"] = df["model_prob_home"] - df["fair_home_prob"]
    df["away_edge"] = df["model_prob_away"] - df["fair_away_prob"]

    current = bankroll
    bank_ts = [bankroll]
    bets = []

    for idx, row in df.iterrows():
        side = None
        if row["home_edge"] > 0:
            side, edge = "home", row["home_edge"]
            vigged_decimal = row["home_decimal_odds"]
        elif row["away_edge"] > 0:
            side, edge = "away", row["away_edge"]
            vigged_decimal = row["away_decimal_odds"]
        else:
            bank_ts.append(current)
            continue

        # Must have valid vigged odds
        if pd.isna(vigged_decimal) or vigged_decimal <= 1.0:
            bank_ts.append(current)
            continue

        # Determine Kelly fraction from strategy
        kelly_frac = None
        for low, high, frac in edge_strategy:
            if low <= edge < high:
                kelly_frac = frac
                break
        if kelly_frac is None:
            bank_ts.append(current)
            continue

        # Kelly stake using VIGGED odds (realistic payout)
        p = row["model_prob_home"] if side == "home" else row["model_prob_away"]
        # Full‑Kelly fraction for the vigged price
        f_full = (p * vigged_decimal - 1) / (vigged_decimal - 1)
        f = max(0.0, min(f_full * kelly_frac, 1.0))
        stake = f * current
        if stake <= 0:
            bank_ts.append(current)
            continue

        won = (row["winning_team"] == 0) if side == "home" else (row["winning_team"] == 1)
        profit = stake * (vigged_decimal - 1) if won else -stake
        current += profit
        bank_ts.append(current)
        bets.append({
            "side": side, "edge": edge, "kelly_frac": kelly_frac,
            "stake": stake, "profit": profit, "won": won
        })

    if not bets:
        return {"final_bankroll": current, "num_bets": 0, "roi_pct": 0.0,
                "win_rate": 0.0, "max_drawdown": 0.0}

    bet_df = pd.DataFrame(bets)
    roi = ((current - bankroll) / bankroll) * 100.0
    win_rate = bet_df["won"].mean()
    bank_series = pd.Series(bank_ts)
    running_max = bank_series.cummax()
    drawdown = (bank_series - running_max) / running_max
    max_dd = drawdown.min()

    return {
        "final_bankroll": round(current, 2),
        "roi_pct": round(roi, 2),
        "num_bets": len(bet_df),
        "home_bets": (bet_df["side"] == "home").sum(),
        "away_bets": (bet_df["side"] == "away").sum(),
        "win_rate": win_rate,
        "mean_edge": bet_df["edge"].mean(),
        "max_drawdown": round(max_dd, 4),
    }

# ═══════════════════════════════════════════════════════════════════════════
# 5. Walk‑forward over multiple splits using DE‑VIGGED FEATURES
# ═══════════════════════════════════════════════════════════════════════════
SPLITS = [0.85, 0.90, 0.95]
all_results = {}   # (split, strategy_name) -> result dict
per_split_top = {}

for split in SPLITS:
    print(f"Processing split = {split:.2f} ...")
    split_idx = int(len(df) * split)
    test_df = df.iloc[split_idx:].reset_index(drop=True)

    # Build feature matrix using de‑vigged features (exactly as the app does)
    COLS_TO_DROP = [
        "winning_team", "date", "home_team", "away_team",
        "home_moneyline", "away_moneyline",
        # Do NOT drop home_decimal_odds / away_decimal_odds;
        # they are needed in simulate_kelly and are not used as features.
    ]
    X = test_df.drop(columns=[c for c in COLS_TO_DROP if c in test_df.columns])
    X = X.select_dtypes(include=[np.number]).fillna(0)
    X = X.reindex(columns=scaler.feature_names_in_, fill_value=0)
    X_scaled = pd.DataFrame(
        scaler.transform(X),
        columns=X.columns,
        index=X.index
    )
    X_final = X_scaled[selected_features]

    # Model prediction (on de‑vigged features!)
    y_pred_proba = model.predict_proba(X_final)[:, 1]

    # Simulate every strategy on this split
    split_res = {}
    for name, edge_strat in strategies:
        sim = simulate_kelly(test_df, y_pred_proba, INITIAL_BANKROLL, edge_strat)
        if sim["num_bets"] > 0:
            split_res[name] = sim
            all_results[(split, name)] = sim

    # Best for this split (subject to drawdown)
    passed = {n: r for n, r in split_res.items() if r["max_drawdown"] >= MAX_DRAWDOWN_LIMIT}
    if passed:
        top_name, top_res = max(passed.items(), key=lambda x: x[1]["roi_pct"])
        per_split_top[split] = (top_name, top_res)
    else:
        per_split_top[split] = None
    print(f"  -> Best on split {split:.2f}: {per_split_top[split]}\n")

# ═══════════════════════════════════════════════════════════════════════════
# 6. Robust selection: must pass drawdown limit on EVERY split
# ═══════════════════════════════════════════════════════════════════════════
strategy_names = [name for name, _ in strategies]
robust_scores = {}
for name in strategy_names:
    splits_data = {}
    present_all = True
    for split in SPLITS:
        if (split, name) in all_results:
            splits_data[split] = all_results[(split, name)]
        else:
            present_all = False
            break
    if not present_all:
        continue
    dd_ok = all(splits_data[split]["max_drawdown"] >= MAX_DRAWDOWN_LIMIT for split in SPLITS)
    if not dd_ok:
        continue
    mean_roi = np.mean([splits_data[s]["roi_pct"] for s in SPLITS])
    robust_scores[name] = mean_roi

if not robust_scores:
    print("No strategy passed the drawdown limit on all splits simultaneously.")
else:
    best_strategy_name = max(robust_scores, key=robust_scores.get)
    best_mean_roi = robust_scores[best_strategy_name]

    print("\n" + "=" * 100)
    print("            ROBUST STRATEGY (de‑vigged features for inference)")
    print("=" * 100)
    print(f"Strategy: {best_strategy_name}")
    for s in SPLITS:
        r = all_results[(s, best_strategy_name)]
        print(f"  Split {s:.2f} | ROI: {r['roi_pct']:+.2f}%  MaxDD: {r['max_drawdown']:.2%}  "
              f"Bets: {r['num_bets']}  Win: {r['win_rate']:.2%}  "
              f"Bank: ${r['final_bankroll']:,.2f}")
    print(f"  Mean ROI across splits: {best_mean_roi:+.2f}%")

    # Top‑5 robust strategies for reference
    top_robust = sorted(robust_scores.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop‑5 robust strategies (by mean ROI):")
    for rank, (name, mean_roi) in enumerate(top_robust, 1):
        print(f"  {rank}. {name}  (mean ROI = {mean_roi:+.2f}%)")

# Print per‑split winners for comparison
print("\n" + "=" * 100)
print("            BEST STRATEGY FOR EACH INDIVIDUAL SPLIT")
print("=" * 100)
for split, data in per_split_top.items():
    if data is not None:
        name, res = data
        print(f"Split {split:.2f}: {name}  ROI={res['roi_pct']:+.2f}%  "
              f"MaxDD={res['max_drawdown']:.2%}  Bets={res['num_bets']}")
    else:
        print(f"Split {split:.2f}: no strategy passed drawdown limit.")

Generated 2440 strategies.

Processing split = 0.85 ...
  -> Best on split 0.85: ('2Bin_min0.035_lo<0.055_fL1.0_fH0.25', {'final_bankroll': 14327.11, 'roi_pct': 43.27, 'num_bets': 334, 'home_bets': np.int64(192), 'away_bets': np.int64(142), 'win_rate': np.float64(0.38922155688622756), 'mean_edge': np.float64(0.05744149619860935), 'max_drawdown': -0.4918})

Processing split = 0.90 ...
  -> Best on split 0.90: ('2Bin_min0.035_lo<0.055_fL1.0_fH0.25', {'final_bankroll': 12160.42, 'roi_pct': 21.6, 'num_bets': 220, 'home_bets': np.int64(133), 'away_bets': np.int64(87), 'win_rate': np.float64(0.38636363636363635), 'mean_edge': np.float64(0.057438079393626544), 'max_drawdown': -0.4492})

Processing split = 0.95 ...
  -> Best on split 0.95: ('2Bin_min0.025_lo<0.055_fL1.0_fH0.25', {'final_bankroll': 20604.39, 'roi_pct': 106.04, 'num_bets': 167, 'home_bets': np.int64(95), 'away_bets': np.int64(72), 'win_rate': np.float64(0.41916167664670656), 'mean_edge': np.float64(0.049777427708722045), 'max_dr

In [ ]:
"""
Walk‑Forward Kelly Simulation on the LAST 15% of the Dataset
------------------------------------------------------------
- Uses only the most recent 15% of games (chronological split)
- Interactive strategy definition (edge bins & Kelly fractions)
- Simulates betting game by game with real vigged odds for payouts
- Edge thresholds must be between 0 and 1 (e.g., 0.02 = 2%)
"""

import numpy as np
import pandas as pd
import joblib

# =============================================================================
# 0. Power‑method de‑vigging (exactly as in original)
# =============================================================================
def power_de_vig(home_implied, away_implied, max_iter=100, tol=1e-12):
    home_implied = np.asarray(home_implied, dtype=float)
    away_implied = np.asarray(away_implied, dtype=float)
    n = len(home_implied)
    fair_home = np.full(n, np.nan)
    fair_away = np.full(n, np.nan)

    for i in range(n):
        h = home_implied[i]
        a = away_implied[i]
        if np.isnan(h) or np.isnan(a) or h <= 0 or a <= 0:
            continue
        if abs(h + a - 1.0) < tol:
            fair_home[i], fair_away[i] = h, a
            continue

        def f(k):
            return h**k + a**k - 1.0
        def fp(k):
            if h > 0 and a > 0:
                return h**k * np.log(h) + a**k * np.log(a)
            return 0.0

        k = 1.0
        converged = False
        for _ in range(max_iter):
            fk = f(k)
            if abs(fk) < tol:
                converged = True
                break
            fpk = fp(k)
            if fpk == 0:
                break
            k_new = k - fk / fpk
            if k_new <= 0:
                k_new = 0.001
            if k_new > 10.0:
                k_new = 10.0
            if abs(k_new - k) < tol:
                k = k_new
                converged = True
                break
            k = k_new

        if not converged:
            lo, hi = 0.001, 1.0
            if f(lo) < 0:
                lo, hi = hi, lo
            for _ in range(max_iter):
                mid = (lo + hi) / 2
                if f(mid) == 0.0 or (hi - lo) / 2 < tol:
                    k = mid
                    converged = True
                    break
                if np.sign(f(mid)) == np.sign(f(lo)):
                    lo = mid
                else:
                    hi = mid
            if not converged:
                total = h + a
                fair_home[i] = h / total if total > 0 else np.nan
                fair_away[i] = a / total if total > 0 else np.nan
                continue

        fair_home[i] = h**k
        fair_away[i] = a**k
    return fair_home, fair_away


# =============================================================================
# 1. Load model bundle and data
# =============================================================================
print("Loading model and data...")
bundle = joblib.load("nba.pkl")
model = bundle["model"]
scaler = bundle["scaler"]
selected_features = bundle["features"]

INITIAL_BANKROLL = 10000.0

# Adjust path if needed
df = pd.read_csv("../data/csv/dataset.csv")


# =============================================================================
# 2. Feature engineering (identical to original)
# =============================================================================
def feature_engineering_app_style(df):
    df = df.copy()

    # Parse moneylines to implied probabilities and vigged decimal odds
    for col in ("home_moneyline", "away_moneyline"):
        if col not in df.columns:
            continue
        clean = (df[col].astype(str)
                 .str.replace("+", "", regex=False)
                 .str.replace(",", "", regex=False)
                 .str.replace(" ", "", regex=False))
        num = pd.to_numeric(clean, errors="coerce")

        # Implied probability (raw, before de‑vigging)
        implied = np.where(
            num >= 100, 100 / (num + 100),
            np.where(
                num <= -100,
                np.abs(num) / (np.abs(num) + 100),
                np.where((num > 1.0) & (num < 100.0), 1.0 / num, np.nan)
            )
        )
        df[f"{col}_implied_prob"] = implied

        # Vigged decimal odds for actual payout
        decimal = np.where(
            num >= 100, num/100.0 + 1.0,
            np.where(
                num <= -100, 100.0 / np.abs(num) + 1.0,
                np.where((num > 1.0) & (num < 100.0), num, np.nan)
            )
        )
        col_name = "home_decimal_odds" if col.startswith("home") else "away_decimal_odds"
        df[col_name] = decimal

    # De‑vig before feature construction
    col_h = "home_moneyline_implied_prob"
    col_a = "away_moneyline_implied_prob"
    if col_h in df.columns and col_a in df.columns:
        h_vals = df[col_h].values
        a_vals = df[col_a].values
        fair_h, fair_a = power_de_vig(h_vals, a_vals)
        df[col_h] = fair_h
        df[col_a] = fair_a

    # Derived features from fair probs
    if all(c in df.columns for c in (col_h, col_a)):
        df["prob_diff"] = df[col_h] - df[col_a]
        df["prob_ratio"] = df[col_h] / (df[col_a] + 1e-5)

    if "home_elo" in df.columns and "away_elo" in df.columns:
        df["elo_diff"] = df["home_elo"] - df["away_elo"]
        df["elo_ratio"] = df["home_elo"] / (df["away_elo"] + 1e-5)

    return df

df = feature_engineering_app_style(df)

# Drop rows where de‑vigged probabilities are missing
df = df.dropna(subset=["home_moneyline_implied_prob", "away_moneyline_implied_prob"]).reset_index(drop=True)

# Sort chronologically for sequential betting
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

# =============================================================================
# 3. TAKE ONLY THE LAST 15% OF THE DATA (test split)
# =============================================================================
split_point = int(0.85 * len(df))
test_df = df.iloc[split_point:].reset_index(drop=True)
print(f"Total games in dataset: {len(df)}")
print(f"Using last 15% as simulation window: {len(test_df)} games")
print(f"Date range of test set: {test_df['date'].min()} to {test_df['date'].max()}\n")


# =============================================================================
# 4. Kelly simulation function (uses vigged odds for payout)
# =============================================================================
def simulate_kelly(test_df, model_probs_away, bankroll, edge_strategy):
    df_sim = test_df.copy()
    df_sim["model_prob_away"] = model_probs_away
    df_sim["model_prob_home"] = 1 - model_probs_away

    df_sim["fair_home_prob"] = df_sim["home_moneyline_implied_prob"]
    df_sim["fair_away_prob"] = df_sim["away_moneyline_implied_prob"]

    df_sim["home_edge"] = df_sim["model_prob_home"] - df_sim["fair_home_prob"]
    df_sim["away_edge"] = df_sim["model_prob_away"] - df_sim["fair_away_prob"]

    current = bankroll
    bank_ts = [bankroll]
    bets = []

    for idx, row in df_sim.iterrows():
        side = None
        if row["home_edge"] > 0:
            side, edge = "home", row["home_edge"]
            vigged_decimal = row["home_decimal_odds"]
        elif row["away_edge"] > 0:
            side, edge = "away", row["away_edge"]
            vigged_decimal = row["away_decimal_odds"]
        else:
            bank_ts.append(current)
            continue

        if pd.isna(vigged_decimal) or vigged_decimal <= 1.0:
            bank_ts.append(current)
            continue

        # Find Kelly fraction for this edge
        kelly_frac = None
        for low, high, frac in edge_strategy:
            if low <= edge < high:
                kelly_frac = frac
                break
        if kelly_frac is None:
            bank_ts.append(current)
            continue

        p = row["model_prob_home"] if side == "home" else row["model_prob_away"]
        # Full Kelly fraction using vigged odds
        f_full = (p * vigged_decimal - 1) / (vigged_decimal - 1)
        f = max(0.0, min(f_full * kelly_frac, 1.0))
        stake = f * current
        if stake <= 0:
            bank_ts.append(current)
            continue

        won = (row["winning_team"] == 0) if side == "home" else (row["winning_team"] == 1)
        profit = stake * (vigged_decimal - 1) if won else -stake
        current += profit
        bank_ts.append(current)

        bets.append({
            "date": row["date"],
            "side": side,
            "edge": edge,
            "kelly_frac": kelly_frac,
            "stake": stake,
            "win": won,
            "profit": profit,
            "decimal_odds": vigged_decimal,
            "model_prob": p,
            "fair_prob": row["fair_home_prob"] if side == "home" else row["fair_away_prob"],
            "bankroll_after": current
        })

    if not bets:
        # Return dictionary with all expected keys, but zero values
        return {
            "final_bankroll": bankroll,
            "roi_pct": 0.0,
            "num_bets": 0,
            "home_bets": 0,
            "away_bets": 0,
            "win_rate": 0.0,
            "mean_edge": 0.0,
            "max_drawdown": 0.0,
            "bets_df": pd.DataFrame()
        }

    bet_df = pd.DataFrame(bets)
    roi = ((current - bankroll) / bankroll) * 100.0
    win_rate = bet_df["win"].mean()
    bank_series = pd.Series(bank_ts)
    running_max = bank_series.cummax()
    drawdown = (bank_series - running_max) / running_max
    max_dd = drawdown.min()

    return {
        "final_bankroll": round(current, 2),
        "roi_pct": round(roi, 2),
        "num_bets": len(bet_df),
        "home_bets": (bet_df["side"] == "home").sum(),
        "away_bets": (bet_df["side"] == "away").sum(),
        "win_rate": win_rate,
        "mean_edge": bet_df["edge"].mean(),
        "max_drawdown": round(max_dd, 4),
        "bets_df": bet_df
    }


# =============================================================================
# 5. Run simulation on the test set (walk‑forward)
# =============================================================================
def run_simulation_on_test_set(edge_strategy, bankroll=INITIAL_BANKROLL):
    # Prepare features for test set only (using same scaler & model)
    COLS_TO_DROP = [
        "winning_team", "date", "home_team", "away_team",
        "home_moneyline", "away_moneyline",
    ]
    X = test_df.drop(columns=[c for c in COLS_TO_DROP if c in test_df.columns])
    X = X.select_dtypes(include=[np.number]).fillna(0)
    X = X.reindex(columns=scaler.feature_names_in_, fill_value=0)
    X_scaled = pd.DataFrame(scaler.transform(X), columns=X.columns, index=X.index)
    X_final = X_scaled[selected_features]

    y_pred_proba = model.predict_proba(X_final)[:, 1]
    return simulate_kelly(test_df, y_pred_proba, bankroll, edge_strategy)


# =============================================================================
# 6. Interactive menu to define the strategy (with validation)
# =============================================================================
def get_strategy_from_user():
    print("\n" + "=" * 60)
    print("DEFINE YOUR BETTING STRATEGY")
    print("=" * 60)
    print("NOTE: Edges are probabilities (0 to 1). Example: 0.02 = 2% edge.")
    print("Strategy types:")
    print("  1. Flat (single edge threshold and Kelly fraction)")
    print("  2. Two‑bin (one breakpoint, two Kelly fractions)")
    print("  3. Three‑bin (two breakpoints, three Kelly fractions)")
    print("  4. Use a predefined robust strategy (min edge 0.02, half‑Kelly)")
    print("  5. Exit\n")

    choice = input("Select strategy type (1-5): ").strip()

    if choice == "5":
        return None

    def validate_edge(val, name):
        try:
            v = float(val)
            if v < 0 or v > 1:
                raise ValueError(f"{name} must be between 0 and 1")
            return v
        except ValueError as e:
            print(f"Error: {e}")
            return None

    if choice == "1":
        min_edge = validate_edge(input("Minimum edge (decimal, e.g., 0.02): "), "Minimum edge")
        if min_edge is None: return None
        kelly_frac = float(input("Kelly fraction (e.g., 0.5 for half‑Kelly): "))
        strategy = [(min_edge, np.inf, kelly_frac)]
        print(f"\nFlat strategy: bet if edge >= {min_edge:.4f}, use {kelly_frac:.2f} Kelly")
        return strategy

    elif choice == "2":
        min_edge = validate_edge(input("Minimum edge for first bin (e.g., 0.0): "), "Minimum edge")
        if min_edge is None: return None
        breakpoint = validate_edge(input("Breakpoint edge (e.g., 0.03): "), "Breakpoint")
        if breakpoint is None: return None
        if breakpoint <= min_edge:
            print("Breakpoint must be > minimum edge. Adjusting to min_edge+0.01")
            breakpoint = min(min_edge + 0.01, 1.0)
        kelly_low = float(input("Kelly fraction for edges < breakpoint: "))
        kelly_high = float(input("Kelly fraction for edges >= breakpoint: "))
        strategy = [(min_edge, breakpoint, kelly_low), (breakpoint, np.inf, kelly_high)]
        print(f"\nTwo‑bin strategy:")
        print(f"  [{min_edge:.4f}, {breakpoint:.4f}) → Kelly {kelly_low:.2f}")
        print(f"  [{breakpoint:.4f}, ∞)      → Kelly {kelly_high:.2f}")
        return strategy

    elif choice == "3":
        min_edge = validate_edge(input("Minimum edge for first bin (e.g., 0.0): "), "Minimum edge")
        if min_edge is None: return None
        bp1 = validate_edge(input("First breakpoint (e.g., 0.025): "), "First breakpoint")
        if bp1 is None: return None
        bp2 = validate_edge(input("Second breakpoint (e.g., 0.045): "), "Second breakpoint")
        if bp2 is None: return None
        if bp1 <= min_edge or bp2 <= bp1:
            print("Breakpoints must be strictly increasing. Adjusting...")
            bp1 = max(bp1, min_edge + 0.005)
            bp2 = max(bp2, bp1 + 0.005)
            if bp2 > 1.0: bp2 = 1.0
        k1 = float(input("Kelly fraction for first  bin (edges < first breakpoint): "))
        k2 = float(input("Kelly fraction for second bin (edges between breakpoints): "))
        k3 = float(input("Kelly fraction for third  bin (edges >= second breakpoint): "))
        strategy = [(min_edge, bp1, k1), (bp1, bp2, k2), (bp2, np.inf, k3)]
        print(f"\nThree‑bin strategy:")
        print(f"  [{min_edge:.4f}, {bp1:.4f}) → Kelly {k1:.2f}")
        print(f"  [{bp1:.4f}, {bp2:.4f}) → Kelly {k2:.2f}")
        print(f"  [{bp2:.4f}, ∞)      → Kelly {k3:.2f}")
        return strategy

    elif choice == "4":
        print("\nUsing predefined robust strategy: min edge 0.02, half‑Kelly")
        return [(0.02, np.inf, 0.5)]

    else:
        print("Invalid choice. Defaulting to flat 0.02 edge, half‑Kelly.")
        return [(0.02, np.inf, 0.5)]


# =============================================================================
# 7. Main execution
# =============================================================================
if __name__ == "__main__":
    strategy = get_strategy_from_user()
    if strategy is None:
        print("Exiting.")
        exit()

    print("\nRunning walk‑forward simulation on the last 15% of the data...")
    results = run_simulation_on_test_set(strategy)

    print("\n" + "=" * 80)
    print("WALK‑FORWARD SIMULATION RESULTS (LAST 15% SPLIT)")
    print("=" * 80)
    print(f"Strategy bins: {strategy}")
    print(f"Test set size: {len(test_df)} games")
    print(f"Initial bankroll:      ${INITIAL_BANKROLL:,.2f}")
    print(f"Final bankroll:        ${results['final_bankroll']:,.2f}")
    print(f"ROI:                   {results['roi_pct']:+.2f}%")
    print(f"Total bets placed:     {results['num_bets']}  (Home: {results['home_bets']}, Away: {results['away_bets']})")
    print(f"Win rate:              {results['win_rate']:.2%}")
    if results['num_bets'] > 0:
        print(f"Mean edge on bets:     {results['mean_edge']:.4f}")
    else:
        print("Mean edge on bets:     N/A (no bets)")
    print(f"Maximum drawdown:      {results['max_drawdown']:.2%}")

    if results['num_bets'] > 0:
        print("\nFirst 10 bets (date, side, edge, stake, win, profit, bankroll):")
        print(results['bets_df'][['date', 'side', 'edge', 'stake', 'win', 'profit', 'bankroll_after']].head(10).to_string(index=False))

        save_log = input("\nSave full bet log to 'bet_log_last15.csv'? (y/n): ").strip().lower()
        if save_log == 'y':
            results['bets_df'].to_csv("bet_log_last15.csv", index=False)
            print("Saved to bet_log_last15.csv")
    else:
        print("\nNo bets were placed with this strategy.")
        print("Check that your edge thresholds are reasonable (e.g., 0.01 to 0.05).")

Loading model and data...
Total games in dataset: 7147
Using last 15% as simulation window: 1073 games
Date range of test set: 2019-01-15 00:00:00 to 2023-01-16 00:00:00


DEFINE YOUR BETTING STRATEGY
NOTE: Edges are probabilities (0 to 1). Example: 0.02 = 2% edge.
Strategy types:
  1. Flat (single edge threshold and Kelly fraction)
  2. Two‑bin (one breakpoint, two Kelly fractions)
  3. Three‑bin (two breakpoints, three Kelly fractions)
  4. Use a predefined robust strategy (min edge 0.02, half‑Kelly)
  5. Exit



Select strategy type (1-5):  2
Minimum edge for first bin (e.g., 0.0):  0.01
Breakpoint edge (e.g., 0.03):  0.025
Kelly fraction for edges < breakpoint:  1
Kelly fraction for edges >= breakpoint:  .25



Two‑bin strategy:
  [0.0100, 0.0250) → Kelly 1.00
  [0.0250, ∞)      → Kelly 0.25

Running walk‑forward simulation on the last 15% of the data...

WALK‑FORWARD SIMULATION RESULTS (LAST 15% SPLIT)
Strategy bins: [(0.01, 0.025, 1.0), (0.025, inf, 0.25)]
Test set size: 1073 games
Initial bankroll:      $10,000.00
Final bankroll:        $7,500.61
ROI:                   -24.99%
Total bets placed:     626  (Home: 323, Away: 303)
Win rate:              42.49%
Mean edge on bets:     0.0426
Maximum drawdown:      -51.64%

First 10 bets (date, side, edge, stake, win, profit, bankroll):
      date side     edge       stake   win      profit  bankroll_after
2019-01-15 home 0.024520  304.142717  True  101.380906    10101.380906
2019-01-16 home 0.025075   51.766565 False  -51.766565    10049.614340
2019-01-16 away 0.018699   97.448051  True   35.435655    10085.049995
2019-01-17 home 0.016777 1261.218747  True   78.826172    10163.876167
2019-01-18 away 0.050761  143.337644 False -143.337644    100